# 05 — Training
**Steps 19–22:** Configure training strategy, handle class imbalance, train the CMAGNN, and plot training curves.

In [ ]:
import os, sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import NeighborLoader
import matplotlib.pyplot as plt
from tqdm import tqdm

# Import model from 04_model
sys.path.insert(0, os.path.join('..', '04_model'))
# Re-define CMAGNN here or import via a shared utils.py
# For portability we redefine it inline:
from torch_geometric.nn import GATConv

DEVICE        = 'cuda' if torch.cuda.is_available() else 'cpu'
PROCESSED_DIR = os.path.join('..', 'data', 'processed')
OUTPUTS_DIR   = os.path.join('..', 'outputs')
os.makedirs(OUTPUTS_DIR, exist_ok=True)

print(f'Device: {DEVICE}')

In [ ]:
# ── Redefine Model (same as 04_model) ────────────────────────

class ModalityBranch(nn.Module):
    def __init__(self, in_dim, hidden_dim, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU()
        )
    def forward(self, x): return self.net(x)

class FusionLayer(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Sequential(nn.Linear(hidden_dim * 2, 2), nn.Softmax(dim=-1))
        self.proj = nn.Linear(hidden_dim, hidden_dim)
    def forward(self, opt_emb, rad_emb):
        w = self.attn(torch.cat([opt_emb, rad_emb], dim=-1))
        return F.relu(self.proj(w[:, 0:1] * opt_emb + w[:, 1:2] * rad_emb))

class CMAGNN(nn.Module):
    def __init__(self, n_optical=5, n_radar=3, hidden_dim=64, n_classes=5, dropout=0.3, gat_heads=4):
        super().__init__()
        self.optical_branch = ModalityBranch(n_optical, hidden_dim, dropout)
        self.radar_branch   = ModalityBranch(n_radar, hidden_dim, dropout)
        self.fusion         = FusionLayer(hidden_dim)
        self.gat1 = GATConv(hidden_dim, hidden_dim // gat_heads, heads=gat_heads, dropout=dropout, concat=True)
        self.gat2 = GATConv(hidden_dim, hidden_dim, heads=1, dropout=dropout, concat=False)
        self.bn1  = nn.BatchNorm1d(hidden_dim)
        self.bn2  = nn.BatchNorm1d(hidden_dim)
        self.drop = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim, n_classes)

    def forward(self, x, edge_index):
        opt = self.optical_branch(x[:, :5])
        rad = self.radar_branch(x[:, 5:])
        h   = self.fusion(opt, rad)
        h   = self.drop(F.relu(self.bn1(self.gat1(h, edge_index))))
        h   = F.relu(self.bn2(self.gat2(h, edge_index)))
        return self.classifier(h)

print('✅ Model class defined.')

In [ ]:
# ── Load Graph Data ───────────────────────────────────────────
graph_data = torch.load(os.path.join(PROCESSED_DIR, 'graph_data.pt'))
graph_data = graph_data.to(DEVICE)
print(graph_data)

In [ ]:
# ── STEP 19-20: Training Config + Class Weights ───────────────

HIDDEN_DIM = 64
DROPOUT    = 0.3
LR         = 0.001
EPOCHS     = 100
BATCH_SZ   = 2048
N_HOPS     = 2
N_CLASSES  = 5

# Inverse-frequency class weights to address class imbalance
y_train = graph_data.y[graph_data.train_mask].cpu().numpy()
class_counts = np.bincount(y_train, minlength=N_CLASSES).astype(float)
class_weights = 1.0 / (class_counts + 1e-6)
class_weights /= class_weights.sum()
class_weights_t = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)

print('Class weights (inverse-frequency):')
CLASS_NAMES = ['Water', 'Trees', 'Crops', 'Built-up', 'Bare Ground']
for i, (name, w) in enumerate(zip(CLASS_NAMES, class_weights)):
    print(f'  {i}: {name:15s} | count: {int(class_counts[i]):>10,} | weight: {w:.6f}')

In [ ]:
# ── STEP 21: Train Model ──────────────────────────────────────

model     = CMAGNN(hidden_dim=HIDDEN_DIM, dropout=DROPOUT).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)
criterion = nn.CrossEntropyLoss(weight=class_weights_t)

# Mini-batch subgraph loader
train_loader = NeighborLoader(
    graph_data,
    num_neighbors=[10] * N_HOPS,
    input_nodes=graph_data.train_mask,
    batch_size=BATCH_SZ,
    shuffle=True
)

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    # ── Training ─────────────────────────────────────────────
    model.train()
    total_loss, correct, total = 0, 0, 0

    for batch in train_loader:
        batch = batch.to(DEVICE)
        optimizer.zero_grad()
        out  = model(batch.x, batch.edge_index)
        loss = criterion(out[:batch.batch_size], batch.y[:batch.batch_size])
        loss.backward()
        optimizer.step()

        preds = out[:batch.batch_size].argmax(dim=1)
        correct += (preds == batch.y[:batch.batch_size]).sum().item()
        total   += batch.batch_size
        total_loss += loss.item()

    train_loss = total_loss / len(train_loader)
    train_acc  = correct / total

    # ── Validation ───────────────────────────────────────────
    model.eval()
    with torch.no_grad():
        out_full  = model(graph_data.x, graph_data.edge_index)
        val_loss  = criterion(out_full[graph_data.val_mask], graph_data.y[graph_data.val_mask]).item()
        val_preds = out_full[graph_data.val_mask].argmax(dim=1)
        val_acc   = (val_preds == graph_data.y[graph_data.val_mask]).float().mean().item()

    scheduler.step(val_loss)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), os.path.join(OUTPUTS_DIR, 'best_model.pth'))

    if epoch % 10 == 0:
        print(f'Epoch {epoch:3d}/{EPOCHS} | '
              f'Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | '
              f'Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}')

print(f'\n✅ Training complete. Best Val Accuracy: {best_val_acc:.4f}')

In [ ]:
# ── STEP 22: Plot Training Curves ────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train Loss', color='steelblue')
axes[0].plot(history['val_loss'],   label='Val Loss',   color='coral')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Accuracy
axes[1].plot(history['train_acc'], label='Train Acc', color='steelblue')
axes[1].plot(history['val_acc'],   label='Val Acc',   color='coral')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training & Validation Accuracy')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
save_path = os.path.join(OUTPUTS_DIR, 'training_curves.png')
plt.savefig(save_path, dpi=150)
plt.show()
print(f'✅ Training curves saved to {save_path}')